# Lesson 0.1: Calculus for ML

**Goal:** be able to derive gradients and Hessians by hand, check them numerically the right way, and use a second-order Taylor expansion to reason about step sizes. Everything in later lessons (backprop, Adam, learning-rate schedules, loss spikes) rests on this.

**Time:** 2-3 days. Watch or read first, then do the exercises without looking anything up beyond the listed sources.

## 1. What to watch and read (all free)

Do these in order. Skip anything you can already explain on a whiteboard.

1. **3Blue1Brown, *Essence of Calculus*** (YouTube playlist): chapters 1-4 (derivatives, power/trig rules, chain and product rule), then chapter 10 (Taylor series). About 1.5 hours.
2. **Khan Academy, Multivariable Calculus** (taught by Grant Sanderson), unit "Derivatives of multivariable functions": partial derivatives, gradient, directional derivatives, "Why the gradient is the direction of steepest ascent". Then unit "Applications of multivariable derivatives": quadratic approximations and the Hessian.
3. ***Mathematics for Machine Learning***, Deisenroth, Faisal & Ong (free PDF at mml-book.github.io), Chapter 5: sections 5.1, 5.2, 5.7 and 5.8. Skim 5.3-5.5 now; you'll do them properly in lesson 0.4.
4. Optional, if something still feels shaky: MIT OCW 18.02SC Multivariable Calculus, Unit 2 (partial derivatives).

## 2. Notes: the parts that matter for ML

**The derivative is a linear approximation.** `f(x + h) ≈ f(x) + f'(x) h`. In many dimensions: `f(w + d) ≈ f(w) + ∇f(w)ᵀ d`. Gradient descent is just "pick `d` to make this linear model go down as fast as possible, for a fixed step length."

**Chain rule.** `(f∘g)'(x) = f'(g(x)) g'(x)`. With vectors, `g: Rⁿ → Rᵐ` and `f: Rᵐ → R`, it becomes `∇ₓ f = J_gᵀ ∇f`, where `J_g` is the `m × n` Jacobian. Backprop is this rule applied right to left, so you only ever multiply a vector by a Jacobian and never build the full Jacobian.

**Gradient.** The vector of partials. The directional derivative along a unit vector `u` is `∇fᵀ u`, which by Cauchy-Schwarz is largest when `u` points along `∇f`. So the gradient points in the direction of steepest ascent, and its length is the slope in that direction.

**Second order.** `f(w + d) ≈ f(w) + gᵀ d + ½ dᵀ H d`, where `H` is the Hessian (the matrix of second partials, symmetric for smooth `f`). For a gradient step `d = -η g`:

`ΔL ≈ -η ‖g‖² + ½ η² gᵀ H g`

The first term is the gain and the second is the curvature penalty. This one formula explains why there is a best learning rate, why too large a rate diverges, and why ill-conditioned problems are slow.

**Quadratic case.** For `f(w) = ½ wᵀ A w` with `A` symmetric positive definite, gradient descent converges if and only if `η < 2 / λ_max(A)`. Its speed is set by the condition number `κ = λ_max / λ_min`. Much of deep learning optimization (momentum, Adam, normalization layers, warmup) is about coping with large and changing `κ`.

**Numerical gradients.** The central difference `(f(x+h) - f(x-h)) / 2h` has truncation error `O(h²)` and rounding error about `ε/h`, where `ε` is machine epsilon. The total error is smallest near `h ≈ ε^(1/3)`. People who pick `h = 1e-8` in float32 get garbage and conclude their correct gradient is wrong.

## 3. Exercises

These are deliberately hard. Rules:
- Derive on paper first, then code it. Hand-derived means **no autograd** inside your functions. Autograd is only used by the check cells.
- Use vectorized torch, with no Python loops over data points.
- Don't edit the check cells. A task counts only when it prints `PASS`.
- Written answers should be short and exact, and math in plain text is fine.

When you're done: File > Download > .ipynb, and upload it in the thread.

In [ ]:
import math
import torch
torch.set_default_dtype(torch.float64)

## Exercise 1: Activation derivatives by hand, then Newton's method

Modern LLMs use SiLU/Swish (inside SwiGLU) and GELU, so you will see these again in Llama and Qwen.

Implement, deriving by hand:
- `d_softplus(x)` for `softplus(x) = log(1 + eˣ)`
- `d_silu(x)` and `d2_silu(x)` for `silu(x) = x·σ(x)`
- `d_gelu_tanh(x)` for the tanh approximation `gelu(x) = 0.5 x (1 + tanh(√(2/π)(x + 0.044715 x³)))`

All of them must be numerically safe for `|x|` up to 1000, so no `inf` or `nan`.

Then:
- SiLU is **not monotonic**. Find the `x` where `silu` reaches its minimum (where `silu' = 0`), and the `x` where `silu'` reaches its minimum (where `silu'' = 0`). Use **Newton's method** written by you, with your own derivatives. For the second root you need `silu'''`, so derive that too, or justify using a secant step instead.

In [ ]:
def d_softplus(x): raise NotImplementedError
def d_silu(x): raise NotImplementedError
def d2_silu(x): raise NotImplementedError
def d_gelu_tanh(x): raise NotImplementedError

def newton(f, df, x0, tol=1e-12, max_iter=100):
    raise NotImplementedError

x_silu_min = None      # x where silu is minimal
x_dsilu_min = None     # x where silu' is minimal

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    x = torch.linspace(-30, 30, 4001, requires_grad=True)
    fs = {"softplus": (torch.nn.functional.softplus, d_softplus),
          "silu": (torch.nn.functional.silu, d_silu),
          "gelu": (lambda z: torch.nn.functional.gelu(z, approximate="tanh"), d_gelu_tanh)}
    for name, (f, df) in fs.items():
        g, = torch.autograd.grad(f(x).sum(), x, create_graph=True)
        assert torch.allclose(df(x.detach()), g.detach(), atol=1e-10), f"{name}' wrong"
        if name == "silu":
            g2, = torch.autograd.grad(g.sum(), x)
            assert torch.allclose(d2_silu(x.detach()), g2, atol=1e-10), "silu'' wrong"
    big = torch.tensor([-1000., -50., 50., 1000.])
    for df in (d_softplus, d_silu, d2_silu, d_gelu_tanh):
        assert torch.isfinite(df(big)).all(), f"{df.__name__} not stable for large |x|"
    assert x_silu_min is not None and abs(d_silu(torch.tensor(x_silu_min))) < 1e-10
    assert x_dsilu_min is not None and abs(d2_silu(torch.tensor(x_dsilu_min))) < 1e-10
    assert -1.4 < x_silu_min < -1.2 and x_dsilu_min < x_silu_min, "found the wrong root"
    print(f"PASS  silu min at x={x_silu_min:.6f}, silu' min at x={x_dsilu_min:.6f}")
_check1()

**Written (1a):** `silu'` goes slightly negative and then back up to 1. What does that mean for gradient flow through a SiLU unit whose input sits at `x = x_silu_min`? Compare with ReLU at `x < 0`.

**Written (1b):** `d_softplus` is a function you already know. Which one, and why does that make `softplus` a "smooth ReLU"?

**Your answer:**

## Exercise 2: Finite differences, done properly

1. Implement `central_diff(f, x, h)` for scalar `f` and scalar `x`.
2. For `f(x) = sin(x)·eˣ` at `x = 1`, compute the absolute error of `central_diff` against the exact derivative for `h = 10^-1 … 10^-15`, once in float64 and once in float32 (cast `x` and `h` to that dtype). Plot `log10(error)` vs `log10(h)`.
3. **Before looking at the plot**, derive the optimal `h` from the error model `error(h) ≈ (|f'''| / 6) h² + ε |f| / h` and implement `predicted_h_opt(dtype)`. Use `torch.finfo(dtype).eps`, and use your own formula for `f'''` at `x = 1`.
4. Implement `num_grad(f, w, h)` for `f: Rⁿ → R` using central differences, vectorized across coordinates by evaluating `f` on a batch of perturbed points. `f` accepts a `(k, n)` batch and returns `(k,)`.

In [ ]:
def central_diff(f, x, h): raise NotImplementedError
def predicted_h_opt(dtype): raise NotImplementedError
def num_grad(f, w, h=1e-5): raise NotImplementedError

# TODO: the error-vs-h experiment and plot (matplotlib)

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    f = lambda x: torch.sin(x) * torch.exp(x)
    exact = math.exp(1) * (math.sin(1) + math.cos(1))
    for dt in (torch.float32, torch.float64):
        hs = [10.0 ** -k for k in torch.arange(1, 15.01, 0.25).tolist()]
        errs = [abs(central_diff(f, torch.tensor(1.0, dtype=dt), torch.tensor(h, dtype=dt)).item() - exact) for h in hs]
        h_emp = hs[min(range(len(hs)), key=errs.__getitem__)]
        hp = float(predicted_h_opt(dt))
        assert 1/30 < hp / h_emp < 30, f"{dt}: predicted h_opt={hp:.1e}, empirical {h_emp:.1e}"
    A = torch.randn(7, 7); A = A @ A.T
    fq = lambda W: 0.5 * ((W @ A) * W).sum(-1) + torch.logsumexp(W, -1)
    w = torch.randn(7, requires_grad=True)
    g, = torch.autograd.grad(fq(w[None])[0], w)
    assert torch.allclose(num_grad(fq, w.detach()), g, atol=1e-7), "num_grad wrong"
    print("PASS")
_check2()

**Written (2a):** Your float32 curve bottoms out at an error around what value? A colleague runs a gradient check in float32 with `h=1e-6` and a tolerance of `1e-6`, and reports that their backward pass is wrong. What do you tell them?

**Your answer:**

## Exercise 3: Gradient and Hessian of L2-regularized logistic regression

Data `X ∈ R^{n×d}`, labels `y ∈ {0,1}ⁿ`, weights `w ∈ Rᵈ`:

`L(w) = (1/n) Σᵢ [ softplus(xᵢᵀw) - yᵢ xᵢᵀw ] + (λ/2) ‖w‖²`

(Check for yourself that this equals the usual binary cross-entropy.)

Derive `∇L` and `∇²L` in matrix form and implement them. Requirements:
- `loss` must be stable for large `|xᵢᵀw|`
- `hess` must **not** build an `n × n` matrix (n can be 10⁶). It's `O(n d²)` time and `O(d²)` extra memory.
- `hvp(w, v)` computes the Hessian-vector product `H v` in `O(n d)`, **without** forming `H`. This is how second-order methods scale to big models.

In [ ]:
def loss(w, X, y, lam): raise NotImplementedError
def grad(w, X, y, lam): raise NotImplementedError
def hess(w, X, y, lam): raise NotImplementedError
def hvp(w, v, X, y, lam): raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _make_lr(n=500, d=12, seed=0):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, d, generator=g)
    wt = torch.randn(d, generator=g)
    y = (torch.rand(n, generator=g) < torch.sigmoid(X @ wt)).double()
    return X, y

def _check3():
    X, y = _make_lr(); lam = 1e-2
    w = torch.randn(12, requires_grad=True)
    f = lambda w: (torch.nn.functional.binary_cross_entropy_with_logits(X @ w, y) + lam / 2 * w @ w)
    assert torch.allclose(loss(w.detach(), X, y, lam), f(w), atol=1e-12), "loss wrong"
    g, = torch.autograd.grad(f(w), w)
    H = torch.autograd.functional.hessian(f, w.detach())
    assert torch.allclose(grad(w.detach(), X, y, lam), g, atol=1e-12), "grad wrong"
    assert torch.allclose(hess(w.detach(), X, y, lam), H, atol=1e-12), "hess wrong"
    v = torch.randn(12)
    assert torch.allclose(hvp(w.detach(), v, X, y, lam), H @ v, atol=1e-12), "hvp wrong"
    assert torch.isfinite(loss(torch.full((12,), 300.), X, y, lam)), "loss not stable"
    print("PASS")
_check3()

**Written (3a):** Prove `∇²L` is positive definite for `λ > 0`. What does that buy you, and what changes when `λ = 0` and the data is linearly separable?

**Your answer:**

## Exercise 4: Taylor expansion, step sizes and conditioning

**4a. Predicting a step.** Using your functions from Exercise 3:
- `predicted_delta(w, eta)`: the second-order prediction of `L(w - η g) - L(w)`
- `eta_star(w)`: the step size that minimizes that prediction (derive it)

**4b. Power iteration.** Implement `lambda_max(matvec, d, iters)` using only matrix-vector products. Use it with `hvp` to get the largest Hessian eigenvalue. This is exactly how papers measure "sharpness" of neural network losses (see Cohen et al. 2021, *Edge of Stability*).

**4c. The 2/λ rule.** On the quadratic `f(w) = ½ wᵀ A w`, run gradient descent for 200 steps with `η = 1.9/λ_max` and `η = 2.1/λ_max`, and return the final `‖w‖` for each.

In [ ]:
def predicted_delta(w, eta, X, y, lam): raise NotImplementedError
def eta_star(w, X, y, lam): raise NotImplementedError
def lambda_max(matvec, d, iters=500): raise NotImplementedError
def gd_on_quadratic(A, w0, eta, steps=200): raise NotImplementedError   # returns final w

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    X, y = _make_lr(n=2000); lam = 1e-1
    w = 0.1 * torch.randn(12, generator=torch.Generator().manual_seed(9))
    g = grad(w, X, y, lam)
    for eta in [0.01, 0.1, eta_star(w, X, y, lam)]:
        actual = loss(w - eta * g, X, y, lam) - loss(w, X, y, lam)
        pred = predicted_delta(w, eta, X, y, lam)
        assert abs(actual - pred) / abs(actual) < 0.05, f"eta={float(eta):.3f}: predicted {float(pred):.5f}, actual {float(actual):.5f}"
    es = eta_star(w, X, y, lam)
    assert all(predicted_delta(w, es, X, y, lam) <= predicted_delta(w, es * s, X, y, lam) for s in (0.9, 1.1)), "eta_star is not the minimizer"
    H = hess(w, X, y, lam)
    lm = lambda_max(lambda v: hvp(w, v, X, y, lam), 12)
    assert abs(lm - torch.linalg.eigvalsh(H).max()) < 1e-6, "power iteration wrong"
    gen = torch.Generator().manual_seed(3)
    Q, _ = torch.linalg.qr(torch.randn(20, 20, generator=gen))
    A = Q @ torch.diag(torch.logspace(-2, 1, 20)) @ Q.T
    L = lambda_max(lambda v: A @ v, 20)
    w0 = torch.randn(20, generator=gen)
    ok, bad = gd_on_quadratic(A, w0, 1.9 / L).norm(), gd_on_quadratic(A, w0, 2.1 / L).norm()
    assert ok < w0.norm() < bad and bad > 1e3, f"2/lambda rule not shown: {ok:.3g}, {bad:.3g}"
    print(f"PASS  lambda_max={float(lm):.4f}, eta*={float(es):.4f}, 2/lambda_max={2/float(lm):.4f}")
_check4()

**Written (4a):** In the quadratic above, `κ = 1000`. With the best fixed step size, how many iterations does GD need to shrink the error along the *smallest*-eigenvalue direction by a factor of `e`? Derive it. What does that suggest about why plain SGD struggles on deep nets, and what problem momentum and Adam are each trying to fix?

**Written (4b):** Compare `eta_star` with `2/λ_max` from your output. Why is the step that's best for *this* gradient different from the stability limit?

**Written (4c):** Prove that the gradient is the direction of steepest ascent (unit-norm directions, Cauchy-Schwarz). Then say why that's not true any more if you measure step length with a different norm, like `‖d‖_∞`. Which optimizer does steepest descent under `‖·‖_∞` resemble?

**Your answer:**

## Done?

Upload the notebook here. I'll review the code and written answers, and then send lesson 0.2 (Linear algebra). If a task took over 3 hours, note it, since that tells me where to slow down.